In [ ]:
import scanpy as sc
import scvelo as scv

bdata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
scv.pl.velocity_embedding_stream(bdata, basis="umap", color="clusters", density=1.5, arrow_size=0.1)

In [ ]:
import numpy as np
from scripts.perturbation_distance import PerturbDistanceSolver

def scale_columns(X):
    return X / np.std(X, axis=0, keepdims=True)

X = bdata.layers["Ms"]
X = scale_columns(X)
V = bdata.layers["velocity"]
V = scale_columns(V)

%time perturb_dist_solver = PerturbDistanceSolver(X, V, 0.3)
%time dist = perturb_dist_solver.pairwise_perturb_distance(rounds=30)

In [ ]:
def first_triangle_violation_vectorized(dist_squared, tol=1e-8):
    # Convert to stable distance matrix
    dist = np.sqrt(np.maximum(dist_squared, 0))
    n = dist.shape[0]

    for k in range(n):
        # Compute dist[i, k] + dist[k, j] as a matrix
        lhs = dist
        rhs = dist[:, k][:, np.newaxis] + dist[k, :][np.newaxis, :]

        # Check for triangle inequality violation
        violation_mask = lhs > rhs + tol
        if np.any(violation_mask):
            i, j = np.argwhere(violation_mask)[0]
            return (i, j, k)

    return None  # No violation found

In [ ]:
# first_triangle_violation_vectorized(dist)

In [ ]:
dist = np.maximum(dist, 0)
dist = np.minimum(dist, dist.T)

In [ ]:
import umap

# Step 1: Fit UMAP using the distance matrix
reducer = umap.UMAP(metric="precomputed", min_dist=0.3)
embedding = reducer.fit_transform(dist)

# Step 2: Save result into AnnData object
bdata.obsm["umap_velocity"] = embedding

# Step 3: Plot UMAP using custom embedding
sc.pl.embedding(bdata, basis="umap_velocity", color="clusters")

In [ ]:
from sklearn.manifold import MDS

mds = MDS(n_components=2, dissimilarity="precomputed", random_state=0)
mds_embedding = mds.fit_transform(dist)

bdata.obsm["mds_velocity"] = mds_embedding
sc.pl.embedding(bdata, basis="mds_velocity", color="clusters")

In [ ]:
from scipy.spatial.distance import cdist
dist_original = cdist(X, X)

mds = MDS(n_components=2, dissimilarity="precomputed", random_state=0)
mds_embedding = mds.fit_transform(dist_original)

bdata.obsm["mds_velocity"] = mds_embedding
sc.pl.embedding(bdata, basis="mds_velocity", color="clusters")